# 🧹 Exploration & nettoyage des données DVF

Notebook d'exploration (Jour 4 — *Le Grand Nettoyage avec Pandas*).
Workflow du data cleaner : **Observer → Diagnostiquer → Corriger → Documenter**.

On charge la table `raw_dvf` depuis la base SQLite `immobilier.db`, on diagnostique
la qualité du dataset, puis on nettoie en justifiant chaque choix.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/immobilier.db')
df = pd.read_sql_query('SELECT * FROM raw_dvf', conn)
df.shape

## 1. Observer
Premiers réflexes : `head()`, `info()`, `describe()`.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Diagnostiquer
Où sont les valeurs manquantes ? Y a-t-il des incohérences de catégories ?
`describe()` révèle déjà des aberrations (valeur_fonciere négative et max démesuré).

In [ ]:
df.isnull().sum()

In [ ]:
df['type_bien'].value_counts()

**Constats :**
- `valeur_fonciere` et `surface_reelle` ont quelques valeurs manquantes → on supprime (essentielles au prix au m²).
- `valeur_fonciere` contient des valeurs négatives et un maximum à 15 milliards € → aberrations à filtrer.
- `surface_terrain` est très lacunaire mais non essentielle → on l'ignore.
- `date_mutation` est en `object` (texte) → à convertir en `datetime`.

## 3. Corriger
Suppression des doublons, des valeurs manquantes essentielles et des aberrations,
puis typage de la date.

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=['valeur_fonciere', 'surface_reelle'])
df = df[(df['surface_reelle'] >= 5) & (df['surface_reelle'] <= 10000)]
df = df[(df['valeur_fonciere'] >= 1000) & (df['valeur_fonciere'] <= 50_000_000)]
df = df[df['type_bien'].isin(['Maison', 'Appartement'])]
df['date_mutation'] = pd.to_datetime(df['date_mutation'], format='%d/%m/%Y', errors='coerce')
df = df.dropna(subset=['date_mutation']).reset_index(drop=True)
df.shape

## 4. Vérifier & documenter
On refait `isnull().sum()` pour confirmer le nettoyage.

> **Pourquoi supprimer plutôt qu'imputer ?** Une transaction sans prix ou sans surface ne
> permet pas de calculer un prix au m² fiable ; l'imputer fausserait l'analyse. On préfère
> la **médiane** à la moyenne pour décrire les prix car elle résiste aux valeurs extrêmes.

In [ ]:
df.isnull().sum()

## 5. Sauvegarder le dataset propre dans la base
(Le pipeline `main.py` automatise ces étapes dans `src/nettoyage.py`.)

In [ ]:
df.assign(date_mutation=df['date_mutation'].astype(str)).to_sql(
    'dvf_clean', conn, if_exists='replace', index=False)
conn.close()
print('dvf_clean sauvegardée :', len(df), 'lignes')